# MPIA Arxiv on Deck 2

Contains the steps to produce the paper extractions.

In [1]:
# Imports
import os
from IPython.display import Markdown, display
from tqdm.notebook import tqdm
import warnings
from PIL import Image 
import re

# requires arxiv_on_deck_2

from arxiv_on_deck_2.arxiv2 import (get_new_papers, 
                                    get_paper_from_identifier,
                                    retrieve_document_source, 
                                    get_markdown_badge)
from arxiv_on_deck_2 import (latex,
                             latex_bib,
                             mpia,
                             highlight_authors_in_list)

# Sometimes images are really big
Image.MAX_IMAGE_PIXELS = 1000000000 

In [2]:
# Some useful definitions.

class AffiliationWarning(UserWarning):
    pass

class AffiliationError(RuntimeError):
    pass

def validation(source: str):
    """Raises error paper during parsing of source file
    
    Allows checks before parsing TeX code.
    
    Raises AffiliationWarning
    """
    check = mpia.affiliation_verifications(source, verbose=True)
    if check is not True:
        raise AffiliationError("mpia.affiliation_verifications: " + check)

        
warnings.simplefilter('always', AffiliationWarning)


def get_markdown_qrcode(paper_id: str):
    """ Generate a qrcode to the arxiv page using qrserver.com
    
    :param paper: Arxiv paper
    :returns: markdown text
    """
    url = r"https://api.qrserver.com/v1/create-qr-code/?size=100x100&data="
    txt = f"""<img src={url}"https://arxiv.org/abs/{paper_id}">"""
    txt = '<div id="qrcode">' + txt + '</div>'
    return txt


def clean_non_western_encoded_characters_commands(text: str) -> str:
    """ Remove non-western encoded characters from a string
    List may need to grow.
    
    :param text: the text to clean
    :return: the cleaned text
    """
    text = re.sub(r"(\\begin{CJK}{UTF8}{gbsn})(.*?)(\\end{CJK})", r"\2", text)
    return text


def get_initials(name: str) -> str:
    """ Get the short name, e.g., A.-B. FamName
    :param name: full name
    :returns: initials
    """
    initials = []
    # account for non western names often in ()
    if '(' in name:
        name = clean_non_western_encoded_characters_commands(name)
        suffix = re.findall(r"\((.*?)\)", name)[0]
        name = name.replace(f"({suffix})", '')
    else:
        suffix = ''
    split = name.split()
    for token in split[:-1]:
        if '-' in token:
            current = '-'.join([k[0] + '.' for k in token.split('-')])
        else:
            current = token[0] + '.'
        initials.append(current)
    initials.append(split[-1].strip())
    if suffix:
        initials.append(f"({suffix})")
    return ' '.join(initials)

## get list of arxiv paper candidates

We use the MPIA mitarbeiter list webpage from mpia.de to get author names
We then get all new papers from Arxiv and match authors

In [3]:
# deal with the author list and edge cases of people that cannot be consistent on their name  

def filter_non_scientists(name: str) -> bool:
    """ Loose filter on expected authorships

    removing IT, administration, technical staff
    :param name: name
    :returns: False if name is not a scientist
    """
    remove_list = ['Licht', 'Binroth', 'Witzel', 'Jordan',
                   'Zähringer', 'Scheerer', 'Hoffmann', 'Düe',
                   'Hellmich', 'Enkler-Scharpegge', 'Witte-Nguy',
                   'Dehen', 'Beckmann', 'Jager', 'Jäger'
                  ]

    for k in remove_list:
        if k in name:
            return False
    return True

def add_author_to_list(author_list: list) -> list:
    """ Add author to list if not already in list
    
    :param author: author name
    :param author_list: list of authors
    :returns: updated list of authors
    """
    add_list = ['T. Henning', 'R. E. Hviding']

    for author in add_list:
        if author not in author_list:
            author_list.append(author)
    return author_list

# get list from MPIA website
# filter for non-scientists (mpia.get_mpia_mitarbeiter_list() does some filtering)
mpia_authors = [k[1] for k in mpia.get_mpia_mitarbeiter_list() if filter_non_scientists(k[1])]
# add some missing author because of inconsistencies in their MPIA name and author name on papers
mpia_authors = add_author_to_list(mpia_authors)

In [4]:
new_papers = get_new_papers()
# add manual references
add_paper_refs = []
new_papers.extend([get_paper_from_identifier(k) for k in add_paper_refs])

def robust_call(fn, value, *args, **kwargs):
    try:
        return fn(value, *args, **kwargs)
    except Exception:
        return value

candidates = []
for paperk in new_papers:
    # Check author list with their initials
    normed_author_list = [robust_call(mpia.get_initials, k) for k in paperk['authors']]
    hl_authors = highlight_authors_in_list(normed_author_list, mpia_authors, verbose=True)
    matches = [(hl, orig) for hl, orig in zip(hl_authors, paperk['authors']) if 'mark' in hl]
    paperk['authors'] = hl_authors
    if matches:
        # only select paper if an author matched our list
        candidates.append(paperk)
print("""Arxiv has {0:,d} new papers today""".format(len(new_papers)))        
print("""          {0:,d} with possible author matches""".format(len(candidates)))

F. Walter  ->  F. Walter  |  ['F. Walter']
B. Pennell  ->  B. Pennell  |  ['B. Pennell']
J. Müller-Horn  ->  J. Müller-Horn  |  ['J. Müller-Horn']
S. Shahaf  ->  S. Shahaf  |  ['S. Shahaf']
H.-W. Rix  ->  H.-W. Rix  |  ['H.-W. Rix']
K. El-Badry  ->  K. El-Badry  |  ['K. El-Badry']
J. Li  ->  J. Li  |  ['J. Li']
X. Lin  ->  X. Lin  |  ['X. Lin']
R. Hoppe  ->  R. Hoppe  |  ['R. Hoppe']
H. Zhang  ->  H. Zhang  |  ['H. Zhang']
K. Jahnke  ->  K. Jahnke  |  ['K. Jahnke']
C. Clontz  ->  C. Clontz  |  ['C. Clontz']
S. Souza  ->  S. Souza  |  ['S. Souza']
N. Neumayer  ->  N. Neumayer  |  ['N. Neumayer']
P. J. Smith  ->  P. J. Smith  |  ['P. J. Smith']
J. E. Davies  ->  J. E. Davies  |  ['J. E. Davies']
A. d. Graaff  ->  A. D. Graaff  |  ['A. D. Graaff']
Arxiv has 93 new papers today
          7 with possible author matches


# Parse sources and generate relevant outputs

From the candidates, we do the following steps:
* get their tarball from ArXiv (and extract data)
* find the main .tex file: find one with \documentclass{...} (sometimes it's non trivial)
* Check affiliations with :func:`validation`, which uses :func:`mpia.affiliation_verifications`
* If passing the affiliations: we parse the .tex source
   * inject sub-documents into the main (flatten the main document)
   * parse structure, extract information (title, abstract, authors, figures...)
   * handles `\graphicspath` if provided
* Generate the .md document.

In [5]:
documents = []
failed = []
for paper in tqdm(candidates):
    # debug crap
    paper['identifier'] = paper['identifier'].lower().replace('arxiv:', '').replace(r'\n', '').strip()
    paper_id = paper['identifier']
    
    folder = f'tmp_{paper_id}'

    try:
        if not os.path.isdir(folder):
            folder = retrieve_document_source(f"{paper_id}", f'tmp_{paper_id}')
        
        try:
            doc = latex.LatexDocument(folder, validation=validation)    
        except AffiliationError as affilerror:
            msg = f"ArXiv:{paper_id:s} is not an MPIA paper... " + str(affilerror)
            failed.append((paper, "affiliation error: " + str(affilerror) ))
            continue
        
        # Hack because sometimes author parsing does not work well
        if (len(doc.authors) != len(paper['authors'])):
            doc._authors = paper['authors']
        else:
            # highlight authors (FIXME: doc.highlight_authors)
            # done on arxiv paper already
            doc._authors = highlight_authors_in_list(
                [get_initials(k) for k in doc.authors], 
                mpia_authors, verbose=True)
        if (doc.abstract) in (None, ''):
            doc._abstract = paper['abstract']
            
        doc.comment = (get_markdown_badge(paper_id) + 
                       "<mark>Appeared on: " + paper['date'] + "</mark> - ")
        if paper['comments']:
            doc.comment += " _" + paper['comments'] + "_"
        
        full_md = doc.generate_markdown_text()
        
        full_md += get_markdown_qrcode(paper_id)
        
        # replace citations
        try:
            bibdata = latex_bib.LatexBib.from_doc(doc)
            full_md = latex_bib.replace_citations(full_md, bibdata)
        except Exception as e:
            print("Issues with the citations")
            print(e)
        
        documents.append((paper_id, full_md))
    except Exception as e:
        warnings.warn(latex.LatexWarning(f"{paper_id:s} did not run properly\n" +
                                         str(e)
                                        ))
        failed.append((paper, "latex error " + str(e)))

  0%|          | 0/7 [00:00<?, ?it/s]

Retrieving document from  https://arxiv.org/e-print/2611.21755


extracting tarball to tmp_2611.21755...

 done.


/opt/hostedtoolcache/Python/3.11.18/x64/lib/python3.11/site-packages/IPython/core/interactiveshell.py:3823: LatexWarning: Multiple tex files.

  exec(code_obj, self.user_global_ns, self.user_ns)
/opt/hostedtoolcache/Python/3.11.18/x64/lib/python3.11/site-packages/IPython/core/interactiveshell.py:3823: LatexWarning: Found documentclass in tmp_2611.21755/XQR30.tex

  exec(code_obj, self.user_global_ns, self.user_ns)
/opt/hostedtoolcache/Python/3.11.18/x64/lib/python3.11/site-packages/arxiv_on_deck_2/latex.py:414: LatexWarning: Latex injecting: 'TabSize.tex' from 'tmp_2611.21755/TabSize.tex'
  warnings.warn(LatexWarning(f"Latex injecting: '{ext}' from '{subsource}'"))
/opt/hostedtoolcache/Python/3.11.18/x64/lib/python3.11/site-packages/arxiv_on_deck_2/latex.py:414: LatexWarning: Latex injecting: 'TabComp.tex' from 'tmp_2611.21755/TabComp.tex'
  warnings.warn(LatexWarning(f"Latex injecting: '{ext}' from '{subsource}'"))
/opt/hostedtoolcache/Python/3.11.18/x64/lib/python3.11/site-packages/a

F. Walter  ->  F. Walter  |  ['F. Walter']
E. Bañados  ->  E. Bañados  |  ['E. Bañados']


... (12 outputs left out)

/tmp/ipykernel_2714/2822249172.py:52: LatexWarning: 2611.22887 did not run properly
list index out of range
  warnings.warn(latex.LatexWarning(f"{paper_id:s} did not run properly\n" +


Retrieving document from  https://arxiv.org/e-print/2611.23150


extracting tarball to tmp_2611.23150...

 done.


### Export the logs

Throughout, we also keep track of the logs per paper. see `logs-{today date}.md` 

In [6]:
import datetime
today = str(datetime.date.today())
logfile = f"_build/html/logs/log-{today}.md"


with open(logfile, 'w') as logs:
    # Success
    logs.write(f'# Arxiv on Deck 2: Logs - {today}\n\n')
    logs.write("""* Arxiv had {0:,d} new papers\n""".format(len(new_papers)))
    logs.write("""    * {0:,d} with possible author matches\n\n""".format(len(candidates)))
    logs.write("## Sucessful papers\n\n")
    display(Markdown("## Successful papers"))
    success = [k[0] for k in documents]
    for candid in candidates:
        if candid['identifier'].split(':')[-1] in success:
            display(candid)
            logs.write(candid.generate_markdown_text() + '\n\n')

    ## failed
    logs.write("## Failed papers\n\n")
    display(Markdown("## Failed papers"))
    failed = sorted(failed, key=lambda x: x[1])
    current_reason = ""
    for paper, reason in failed:
        if 'affiliation' in reason:
            color = 'green'
        else:
            color = 'red'
        data = Markdown(
                paper.generate_markdown_text() + 
                f'\n|<p style="color:{color:s}"> **ERROR** </p>| <p style="color:{color:s}">{reason:s}</p> |'
               )
        if reason != current_reason:
            logs.write(f'### {reason:s} \n\n')
            current_reason = reason
        logs.write(data.data + '\n\n')
        
        # only display here the important errors (all in logs)
        # if color in ('red',):
        display(data)

## Successful papers


|||
|---:|:---|
| [![arXiv](https://img.shields.io/badge/arXiv-2611.21755-b31b1b.svg)](https://arxiv.org/abs/2611.21755) | **XQR30: ALMA Observations Reveal the Diverse Structure of Cold Gas within z>6 Quasar Host Galaxies**  |
|| M. Neeleman, et al. -- incl., <mark>F. Walter</mark> |
|*Appeared on*| *2026-11-18*|
|*Comments*| *Accepted for publication in ApJ*|
|**Abstract**|            We present 0.25(~1.5kpc) ALMA observations of the [CII] and dust continuum emission from the host galaxies of 24 luminous z~6 quasars. Together with archival data from 17 quasars, this constitutes a nearly complete sample of all quasars with J-band magnitudes of J_AB<20.0 and redshifts between 5.8 and 6.6, which are the limits defined in the VLT/XSHOOTER legacy survey, XQR-30. For the full sample of 41 quasars, we find that 39(95%) of the quasar host galaxies are detected in [CII] emission with [CII] luminosities ranging between (0.3 and 9.2)x10^9Lsun, and 39(95%) of the quasar host galaxies are detected in continuum emission with continuum flux densities ranging between 0.22 and 16.3mJy. We detect eight [CII]-emitting companion galaxies, which is consistent with the expected number of galaxies based on clustering results. We find that z~6 quasar host galaxies have a range of morphologies, and by defining a new parameter, the kinematic position angle alignment parameter, Delta_PA_{kin-CII}, we show that most quasar host galaxies deviate from a standard disk model. This suggests that [CII] emission probes structures beyond simple disk rotation, such as bars, tidally disturbed gas, etc. We show that the dust emission is more compact than the [CII] emission independent of the morphology of the galaxy. Finally, we provide rough dynamical mass estimates within twice the [CII] half-light radius for the majority of the galaxies, and show that the average dynamical mass within the inner few kpc is consistent with estimates of the average mass in stars, gas and the SMBH.         |


|||
|---:|:---|
| [![arXiv](https://img.shields.io/badge/arXiv-2611.21800-b31b1b.svg)](https://arxiv.org/abs/2611.21800) | **Brown dwarfs and stars are dynamically indistinguishable as companions in AU-separation binaries**  |
|| <mark>B. Pennell</mark>, <mark>J. Müller-Horn</mark>, <mark>S. Shahaf</mark>, <mark>H.-W. Rix</mark>, <mark>K. El-Badry</mark>, <mark>J. Li</mark> |
|*Appeared on*| *2026-11-18*|
|*Comments*| *Submitted to A&A November 16th, 2026. Comments welcome*|
|**Abstract**|            Brown dwarfs occupy the mass range between stars and planets, populations whose dominant formation channels differ, which leave measurable differences in the orbital eccentricity and host-star metallicity distributions. We assemble a sample of astrometric binaries with main sequence primaries and brown dwarf companions at $\mathcal{O}$(AU) separations from $\textit{Gaia}$ DR3. The observable photocenter motions for such binaries are degenerate between a genuinely low-mass companion and a near-equal-mass binary. We break this degeneracy using $\textit{Gaia}$ XP spectrophotometry and follow-up FEROS spectroscopy, which removes removes ${\sim}70\%$ of initial low-mass-function candidates as likely near-twin binaries and leaves residual contamination of only a few percent. The resulting 368 systems, with masses and metallicities determined from the XP spectra and a selection function forward-modelled with $\texttt{Gaiamock}$, form the largest homogeneous sample of brown dwarf companions with astrometric orbits. We find the eccentricity distribution to be well described by a normal distribution. This is comparable to eccentricities of stellar companions at the same periods and higher than those of giant planets, with no significant dependence on companion mass ($\alpha=-0.13\pm0.09$) or period ($\beta=0.10\pm0.05$). The host-star metallicity distribution is likewise stellar-like, extending to [M/H]$=-0.5$, in contrast to the predominantly super-solar metallicity hosts of giant planets. After accounting for selection, the companion-mass distribution over all brown dwarf masses is consistent with an extension of stellar-binary demographics. Brown dwarf companions at AU separations down to at least $20\,M_J$ therefore appear to form predominantly through a binary-star-like channel. A possible transition to planet-like formation at lower masses cannot be located with the present sample.         |


|||
|---:|:---|
| [![arXiv](https://img.shields.io/badge/arXiv-2611.22473-b31b1b.svg)](https://arxiv.org/abs/2611.22473) | **A branchless single-precision Voigt approximation for spectral synthesis**  |
|| <mark>R. Hoppe</mark> |
|*Appeared on*| *2026-11-18*|
|*Comments*| *Accepted for publication in J. Quant. Spectrosc. Radiat. Transf. 10 pages, 2 figures, 5 tables. Fortran module implementing the QLG members included as ancillary file*|
|**Abstract**|            The Voigt function is evaluated repeatedly in spectral synthesis, and several widely used implementations carry an absolute error floor. We construct QLG, a family of branchless, single-precision approximations, separable in the damping parameter $a$. QLG is named for its three ingredients: Lorentzian and Gaussian generators $L$ and $G$, and an envelope $q$ that multiplies every fitted correction and vanishes at both endpoints. The leading Lorentzian wing amplitude is fixed analytically. Three members reach observed maximum full-profile relative errors of $6.5\times10^{-3}$, $9.2\times10^{-4}$, and $7\times10^{-5}$ over $a\in[10^{-9},10^{9}]$ and $|v|\le10^{5}$. Extended-domain and randomized checks support these error levels. For the tested SIU, MOOG, and Humlicek w4 implementations, the relative error instead approaches 100% towards small values of the damping parameter. We compare synthesis-weighted costs on an Apple M1 and an AMD EPYC 9554 using gfortran 15, and propagate errors through solar-dwarf, red-supergiant, and M-dwarf syntheses. The least expensive member costs 0.98-1.05 times as much as the tabulated Synspec kernel on the M1 and 1.11-1.20 times as much on the EPYC, with a maximum flux error smaller by a factor of 3.6. The intermediate member costs less than every tested arithmetic kernel and holds the flux error below $10^{-3}$. The most accurate member costs 1.6-3.0 times less than w4 and gives 3.4-5.0 times smaller maximum flux errors in the tested syntheses.         |


|||
|---:|:---|
| [![arXiv](https://img.shields.io/badge/arXiv-2611.23150-b31b1b.svg)](https://arxiv.org/abs/2611.23150) | **oMEGACat-JWST I.: Project description and photometric data release**  |
|| M. Häberle, et al. -- incl., <mark>C. Clontz</mark>, <mark>S. Souza</mark>, <mark>N. Neumayer</mark>, <mark>P. J. Smith</mark>, <mark>J. E. Davies</mark>, <mark>A. d. Graaff</mark> |
|*Appeared on*| *2026-11-18*|
|*Comments*| *16 pages, 11 Figures; Submitted to Astronomy and Astrophysics*|
|**Abstract**|            Omega Centauri (NGC 5139, $\omega$ Cen) is the most massive globular cluster in the Milky Way and the likely stripped nucleus of an accreted dwarf galaxy. Extensive HST observations have uncovered many of its peculiar features, including complex subpopulations and kinematic evidence for an intermediate-mass black hole via central fast-moving stars. High-precision astrometric follow-up is essential to constrain the orbits of these stars. We present oMEGACat-JWST, a deep photometric and astrometric catalog of $\omega$ Cen's core and surroundings that extends our previous oMEGACat efforts to near-infrared wavelengths. The superior sensitivity and angular resolution of the James Webb Space Telescope (JWST) provide significant astrometric and photometric improvements, particularly for faint stars. Our catalog is based on a custom reduction of NIRCam and NIRISS imaging across three JWST programs extending out to two half-light radii (10'). In total, our catalog contains approximately 677 000 sources. In the region overlapping with oMEGACat, we achieve high relative completeness (> 90% of HST sources recovered) while detecting over 86 000 new sources below HST's detection limit. Astrometric tests demonstrate a single-image precision as good as 0.5% of a NIRCam pixel (0.15 mas) at mF115W=18; at mF115W=23 the precision is still 1 mas. Color-magnitude diagrams based on our data show the well-known split between the primordial Helium and the Helium enriched populations and are particularly sensitive to light-element (CNO) variations of faint stars. oMEGACat-JWST provides the most complete photometric catalog of $\omega$ Cen's core to date. Combined with HST data, it enables multi-wavelength subpopulation analyses. Together with future monitoring observations, its high astrometric precision will improve our understanding of the kinematics of the central fast-moving stars in $\omega$ Cen.         |

## Failed papers


|||
|---:|:---|
| [![arXiv](https://img.shields.io/badge/arXiv-2611.22316-b31b1b.svg)](https://arxiv.org/abs/2611.22316) | **Lord/Egg of the LRDs: One geometry for its many faces**  |
|| P. Madau, et al. -- incl., <mark>X. Lin</mark> |
|*Appeared on*| *2026-11-18*|
|*Comments*| *15 pages, 9 figures, submitted*|
|**Abstract**|            Little red dots (LRDs) combine V-shaped ultraviolet-optical continua, broad Balmer emission, and weak X-rays. The nearby analog SDSS J102530.29+140207.3, known as Lord/Egg, offers detailed spectroscopy and polarimetry. We test one active galactic nucleus (AGN) geometry by fitting its ultraviolet-to-infrared continuum with an inclination-dependent accretion-disk spectral energy distribution (SED), young stars, and a clumpy dusty screen. The same SED is used in Cloudy calculations of a radially stratified, virialized broad-line region (BLR). The fit favors log10(M_BH/M_sun) about 7.4, highly super-Eddington accretion, inclination above 70 degrees, and line-of-sight cloud extinction of about 3 mag. It also predicts a 2 keV luminosity below the Chandra limit. A BLR whose density rises from 10^9 to 10^11 cm^-3 near the sublimation radius reproduces the broad H-alpha profile without a separate intermediate-width Gaussian. For cloud columns near 10^23 cm^-2, it also matches the different Balmer-line widths, exponential-like wings, and Balmer decrement without electron scattering. The redshifted H-beta-H-delta absorber lacks an H-alpha counterpart, requiring velocity-coherent H-alpha re-emission into the absorption trough. Cool outer clumps explain K I and saturated Na I absorption, while the geometry recovers the continuum and H-alpha polarization and their position-angle offset. Ca II triplet absorption requires localized, non-radiatively heated, warm dust-poor zones near the sublimation front. He I absorption traces a fast, about 500 km s^-1, low-column ionized BLR skin. All projected absorber velocities remain below the local escape speeds, consistent with a bound failed BLR wind. Thus, a dusty, high-inclination accretion system can explain Lord/Egg within one connected geometry.         |
|<p style="color:green"> **ERROR** </p>| <p style="color:green">affiliation error: mpia.affiliation_verifications: 'Heidelberg' keyword not found.</p> |


|||
|---:|:---|
| [![arXiv](https://img.shields.io/badge/arXiv-2611.22487-b31b1b.svg)](https://arxiv.org/abs/2611.22487) | **Dual and Binary AGN at Cosmic Dawn: The need for a JWST Program to resolve the merger channel of Early Black Hole Growth**  |
|| F. Mannucci, et al. -- incl., <mark>H. Zhang</mark> |
|*Appeared on*| *2026-11-18*|
|*Comments*| *MCT Concept White Paper, Genesis initiative, comments are welcome*|
|**Abstract**|            The discovery with JWST of accreting black holes (BHs) at z=4-9 in numbers far exceeding pre-JWST expectations, many of them overmassive relative to their hosts and linked to the enigmatic Little Red Dots (LRDs), raises the question of how early BHs form and grow. Hierarchical assembly predicts that galaxy mergers drive gas to galactic nuclei and bring BHs together, leaving a direct observational signature: dual AGN at tens of kpc separations and sub-kpc to few-kpc "duets" in post-merger galaxies, which are the progenitors of the low-frequency GW sources targeted by pulsar timing arrays and LISA. JWST has opened the window to study these systems at z>3. However, these early discoveries are very limited number and come from heterogeneous, largely serendipitous samples, so these systems can hardly be used for robust statistical studies. Within the Genesis initiative We propose a uniformly selected census of dual AGN and their hosts from cosmic noon to cosmic dawn, as a science case for the Genesis initiative. Our tiered strategy uses the survey imaging and spectroscopy for candidate selection at no extra cost, followed by NIRSpec IFU confirmation, multi-band NIRCam (and MIRI/ALMA) host characterisation, and multi-epoch revisits. Confirming and characterising about 100 pairs requires of order 1000 hr, well beyond the ceiling of standard TAC programs, and demands multi-cycle continuity. The census will deliver pair fractions as functions of redshift, separation and luminosity, BH masses and mass ratios, host properties and environments, and the empirical anchor needed to turn merger models into realistic LISA rate forecasts.         |
|<p style="color:green"> **ERROR** </p>| <p style="color:green">affiliation error: mpia.affiliation_verifications: 'Heidelberg' keyword not found.</p> |


|||
|---:|:---|
| [![arXiv](https://img.shields.io/badge/arXiv-2611.22887-b31b1b.svg)](https://arxiv.org/abs/2611.22887) | **Euclid: Quick Data Release (Q1) -- The evolution of the radio-loud AGN fraction-stellar mass relation to $z\sim3$ in the Euclid Deep Field North**  |
|| S. Visser, et al. -- incl., <mark>K. Jahnke</mark> |
|*Appeared on*| *2026-11-18*|
|*Comments*| **|
|**Abstract**|            The incidence of radio-loud active galactic nuclei (RLAGN) and their energetic output via AGN feedback are crucial in understanding the evolution of galaxies and the large-scale structure of the Universe. In the local Universe, the incidence of RLAGN has been shown to correlate strongly with host galaxy properties, particularly stellar mass. Using the large-area survey capabilities provided by Euclid and the LOw Frequency ARray (LOFAR), we present the first study of the RLAGN incidence--stellar mass relation to $z=3.0$. We perform a stringent analysis accounting for cross-matching reliability and stellar-mass incompleteness. In addition, we test the robustness of our results against AGN and host galaxy classification. This yields a sample of $3\,154$ RLAGN in the Euclid Deep Field North, comprising $2\,803$ low-excitation radio galaxies (LERGs). We find that the LERG incidence as a function of stellar mass and radio luminosity follows a tight correlation over more than two orders of magnitude in stellar mass to $z=3.0$. The slope of the power-law relation decreases from $\gamma\approx2.5$ in the local Universe to $\gamma\approx1.0$ at $z=3.0$, indicating a relative increase in low-mass LERGs at earlier epochs. Separating the LERG population into actively star-forming and quiescent galaxies reveals that quiescent LERGs show a much steeper relation between incidence and stellar mass, with $\gamma\approx2.5$ compared to star-forming LERGs that show $\gamma\approx1.5$. However, the relation seems to be independent of whether galaxies are star-forming or quiescent at $z\geq2$, where both populations show relatively flat relations with $\gamma\approx1.0$. Our results suggest an evolutionary picture in which the LERG population transitions from cold-mode, merger-dominated accretion at earlier epochs to a scenario in which systems require a massive hot halo to fuel the AGN in the local Universe.         |
|<p style="color:red"> **ERROR** </p>| <p style="color:red">latex error list index out of range</p> |

## Export documents

We now write the .md files and export relevant images

In [7]:
def export_markdown_summary(md: str, md_fname:str, directory: str):
    """Export MD document and associated relevant images"""
    import os
    import shutil
    import re

    if (os.path.exists(directory) and not os.path.isdir(directory)):
        raise RuntimeError(f"a non-directory file exists with name {directory:s}")

    if (not os.path.exists(directory)):
        print(f"creating directory {directory:s}")
        os.mkdir(directory)

    fig_fnames = (re.compile(r'\[Fig.*\]\((.*)\)').findall(md) + 
                  re.compile(r'\<img src="([^>\s]*)"[^>]*/>').findall(md))
    print("found figures", fig_fnames)
    for fname in fig_fnames:
        if 'http' in fname:
            # No need to copy online figures
            continue
        if not os.path.exists(fname):
            print("file not found", fname)
            continue
        print("copying ", fname, "to", directory)
        destdir = os.path.join(directory, os.path.dirname(fname))
        destfname = os.path.join(destdir, os.path.basename(fname))
        try:
            os.makedirs(destdir)
        except FileExistsError:
            pass
        shutil.copy(fname, destfname)
    with open(os.path.join(directory, md_fname), 'w') as fout:
        fout.write(md)
    print("exported in ", os.path.join(directory, md_fname))
    [print("    + " + os.path.join(directory,fk)) for fk in fig_fnames]

In [8]:
for paper_id, md in documents:
    export_markdown_summary(md, f"{paper_id:s}.md", '_build/html/')

found figures ['tmp_2611.21755/./ciimap.png', 'tmp_2611.21755/./pahist.png', 'tmp_2611.21755/./contmap.png']
copying  tmp_2611.21755/./ciimap.png to _build/html/
copying  tmp_2611.21755/./pahist.png to _build/html/
copying  tmp_2611.21755/./contmap.png to _build/html/
exported in  _build/html/2611.21755.md
    + _build/html/tmp_2611.21755/./ciimap.png
    + _build/html/tmp_2611.21755/./pahist.png
    + _build/html/tmp_2611.21755/./contmap.png
found figures ['tmp_2611.21800/./figures/fiducial_model_comparison.png', 'tmp_2611.21800/./figures/sample_plot.png', 'tmp_2611.21800/./figures/metallicity_distribution.png']
copying  tmp_2611.21800/./figures/fiducial_model_comparison.png to _build/html/
copying  tmp_2611.21800/./figures/sample_plot.png to _build/html/
copying  tmp_2611.21800/./figures/metallicity_distribution.png to _build/html/
exported in  _build/html/2611.21800.md
    + _build/html/tmp_2611.21800/./figures/fiducial_model_comparison.png
    + _build/html/tmp_2611.21800/./figures

## Display the papers

Not necessary but allows for a quick check.

In [9]:
[display(Markdown(k[1])) for k in documents];

<div class="macros" style="visibility:hidden;">
$\newcommand{\ensuremath}{}$
$\newcommand{\xspace}{}$
$\newcommand{\object}[1]{\texttt{#1}}$
$\newcommand{\farcs}{{.}''}$
$\newcommand{\farcm}{{.}'}$
$\newcommand{\arcsec}{''}$
$\newcommand{\arcmin}{'}$
$\newcommand{\ion}[2]{#1#2}$
$\newcommand{\textsc}[1]{\textrm{#1}}$
$\newcommand{\hl}[1]{\textrm{#1}}$
$\newcommand{\footnote}[1]{}$
$\newcommand{\CII}{[\ion{C}{2}]}$
$\newcommand{\kms}{km s^{-1}}$</div>



<div id="title">

# $\Large$  XQR30: ALMA Observations Reveal the Diverse Structure of Cold Gas within $\bm{z \gtrsim 6}$ Quasar Host Galaxies.

</div>
<div id="comments">

[![arXiv](https://img.shields.io/badge/arXiv-2611.21755-b31b1b.svg)](https://arxiv.org/abs/2611.21755)<mark>Appeared on: 2026-11-18</mark> -  _Accepted for publication in ApJ_

</div>
<div id="authors">

M. Neeleman, et al. -- incl., <mark>F. Walter</mark>, <mark>E. Bañados</mark>

</div>
<div id="abstract">

**Abstract:** We present  0 $\farcs$ 25 ( $\approx$ 1.5 kpc) ALMA observations of the $\CII$ and dust continuum emission from the host galaxies of 24 luminous $z \gtrsim 6$ quasars. Together with archival data from 17 quasars, this constitutes a nearly complete sample of all quasars with $J$ -band magnitudes of $J_{\rm AB} < 20.0$ and redshifts between 5.8 and 6.6, which are the limits defined in the VLT/XSHOOTER legacy survey, XQR-30. For the full sample of 41 quasars, we find that 39 (95 \% ) of the quasar host galaxies are detected in $\CII$ emission with $\CII$ luminosities ranging between (0.3 and 9.2) $\times 10^9 L_\odot$ , and 39 (95 \% )  of the quasar host galaxies are detected in continuum emission with continuum flux densities ranging between 0.22 and 16.3 mJy. We detect eight $\CII$ -emitting companion galaxies, which is consistent with the expected number of galaxies based on clustering results. We find that $z \gtrsim 6$ quasar host galaxies have a range of morphologies, and by defining a new parameter, the kinematic position angle alignment parameter, $\Delta$ PA $_{\rm kin-CII}$ , we show that most quasar host galaxies deviate from a standard disk model. This suggests that $\CII$ emission probes structures beyond simple disk rotation, such as bars, tidally disturbed gas, etc. We show that the dust emission is more compact than the $\CII$ emission independent of the morphology of the galaxy. Finally, we provide rough dynamical mass estimates within twice the $\CII$ half-light radius for the majority of the galaxies, and show that the average dynamical mass within the inner few kpc is consistent with estimates of the average mass in stars, gas and the SMBH.

</div>

<div id="div_fig1">

<img src="tmp_2611.21755/./ciimap.png" alt="Fig5" width="100%"/>

**Figure 5. -** $\CII$ images of the quasar host galaxies in the enlarged XQR30 sample. The sample is subdivided by the 24 galaxies that were newly observed as part of this survey, and those with archival data on the $\CII$ emission line. The contours start at 3$\sigma$ and increase in powers of $\sqrt{2}$. The beam of the observations is shown in the bottom left insert of each panel. The black cross marks the optical position of the quasar, and the black dashed circle is the region used for analysis of the $\CII$ emission line. The cyan ellipse marks the beam-deconvolved half-light region of the $\CII$ emission, which is plotted for those galaxies with robust size constraints.
 (*fig:ciimap*)

</div>
<div id="div_fig2">

<img src="tmp_2611.21755/./pahist.png" alt="Fig3" width="100%"/>

**Figure 3. -** Histograms of the kinematic position angle alignment parameters, $\Delta$PA$_{\rm kin-CII}$ and $\Delta$PA$_{\rm kin-cont}$. Only quasars with an ellipticity greater than 0.2 are shown in order to have reliable estimates of the position angle of the emission. The sample is subdivided by their morphology.  The black line shows the distribution of the kinematic position angle alignment parameters for a sample of mock disk galaxies at similar resolution and S/N as the data, showing that $\Delta$PA$_{\rm kin-CII}$ and $\Delta$PA$_{\rm kin-cont}$ are below 30◦ (dashed line) for pure disk rotation. This figure shows that $\Delta$PA$_{\rm kin-CII}$ is a good indicator of potential disk disturbances without performing a full kinematic analysis.
 (*fig:pahist*)

</div>
<div id="div_fig3">

<img src="tmp_2611.21755/./contmap.png" alt="Fig8" width="100%"/>

**Figure 8. -** Dust continuum images of the quasar host galaxies in the enlarged XQR30 sample. The sample is subdivided by the 24 galaxies that were newly observed as part of this survey, and those with archival data on the dust continuum. The contours start at 3$\sigma$ and increase in powers of 2. The synthesized beam of the observations is shown in the bottom left insert of each panel. The black cross marks the optical position of the quasar, and the black dashed circle is the region used for analysis of the dust continuum emission. The cyan ellipse marks the beam-deconvolved half-light region of the dust continuum emission, which is plotted for those galaxies with robust size constraints.
 (*fig:contmap*)

</div><div id="qrcode"><img src=https://api.qrserver.com/v1/create-qr-code/?size=100x100&data="https://arxiv.org/abs/2611.21755"></div>

<div class="macros" style="visibility:hidden;">
$\newcommand{\ensuremath}{}$
$\newcommand{\xspace}{}$
$\newcommand{\object}[1]{\texttt{#1}}$
$\newcommand{\farcs}{{.}''}$
$\newcommand{\farcm}{{.}'}$
$\newcommand{\arcsec}{''}$
$\newcommand{\arcmin}{'}$
$\newcommand{\ion}[2]{#1#2}$
$\newcommand{\textsc}[1]{\textrm{#1}}$
$\newcommand{\hl}[1]{\textrm{#1}}$
$\newcommand{\footnote}[1]{}$
$\newcommand{\n}[1]{\textbf{[NOTE: #1]}}$
$\newcommand{\nc}[1]{\textbf{(CITE: #1)}}$
$\newcommand{\gaia}{\textit{Gaia}~}$
$\newcommand{\tgaia}{\textit{Gaia}}$
$\newcommand{\fmast}{f_{m,\text{ast}}}$
$\newcommand{\mjup}{ M_\text{Jup}}$
$\newcommand{\msun}{ M_\odot}$
$\newcommand{\di}{\text{d}}$
$\newcommand{\bigp}{\text{Pr}}$
$\newcommand{\prob}[2]{\text{Pr}(#1 \mid #2)}$
$\newcommand{\likelihood}[2]{\mathcal{L}(#1 \mid #2)}$</div>



<div id="title">

# Brown dwarfs and stars are dynamically indistinguishable as companions in AU-separation binaries

</div>
<div id="comments">

[![arXiv](https://img.shields.io/badge/arXiv-2611.21800-b31b1b.svg)](https://arxiv.org/abs/2611.21800)<mark>Appeared on: 2026-11-18</mark> -  _Submitted to A&A November 16th, 2026. Comments welcome_

</div>
<div id="authors">

<mark>B. Pennell</mark>, <mark>J. Müller-Horn</mark>, <mark>S. Shahaf</mark>, <mark>H.-W. Rix</mark>, <mark>K. El-Badry</mark>, <mark>J. Li</mark>

</div>
<div id="abstract">

**Abstract:** Brown dwarfs occupy the mass range between stars and planets ( ${\sim}\!13$ -- $80 M_\mathrm{Jup}$ ), populations whose dominant formation channels differ, which leave measurable differences in the orbital eccentricity and host-star metallicity distributions. We assemble a sample of astrometric binaries with main sequence primaries and brown dwarf companions at $\mathcal{O}(\mathrm{AU})$ separations from $_ Gaia_$ DR3. The observable photocenter motions for such binaries are degenerate between a genuinely low-mass companion and a near-equal-mass binary. We break this degeneracy using $_ Gaia_$ XP spectrophotometry and follow-up FEROS spectroscopy, which removes removes ${\sim}70\%$ of initial low-mass-function candidates as likely near-twin binaries and leaves residual contamination of only a few percent. The resulting 368 systems, with masses and metallicities determined from the XP spectra and a selection function forward-modelled with ${\tt Gaiamock}$ , form the largest homogeneous sample of brown dwarf companions with astrometric orbits. We find the eccentricity distribution to be well described by a normal distribution with $\bar{e}=0.46\pm0.02$ and $\sigma_e=0.16\pm0.01$ . This is comparable to eccentricities of stellar companions at the same periods and higher than those of giant planets, with no significant dependence on companion mass ( $\alpha=-0.13\pm0.09$ ) or period ( $\beta=0.10\pm0.05$ ). The host-star metallicity distribution is likewise stellar-like, extending to $\mathrm{[M/H]}\lesssim-0.5$ , in contrast to the predominantly super-solar metallicity hosts of giant planets. After accounting for selection, the companion-mass distribution over all brown dwarf masses is consistent with an extension of stellar-binary demographics. Brown dwarf companions at AU separations down to at least ${\sim}20 M_\mathrm{Jup}$ therefore appear to form predominantly through a binary-star-like channel. A possible transition to planet-like formation at lower masses cannot be located with the present sample.

</div>

<div id="div_fig1">

<img src="tmp_2611.21800/./figures/fiducial_model_comparison.png" alt="Fig6" width="100%"/>

**Figure 6. -** Fiducial models propagated through {\tt Gaiamock} compared to the MS-BD sample, demonstrating that the fiducial models generate good fits to the observed population parameters when selection is accounted for. Bin heights are set to fraction of sources of that sample in the bin such that the areas under each histogram is the same. Error bars on the observed distributions are computed with Poisson uncertainties with an additional 5\% uncertainty to account for remaining spurious solutions (Sect. \ref{sec:Contamination}). Error bars on the synthetic sample are estimated via bootstrapping: samples are drawn from the whole sample of binaries such that they have the same number of NSS sources $1000$ times, the error bars represent 16th and 84th percentile in each bin. A curious lack of observed sources near the DR3 baseline is currently unexplainable. (*fig:FiducialComparison*)

</div>
<div id="div_fig2">

<img src="tmp_2611.21800/./figures/sample_plot.png" alt="Fig5" width="100%"/>

**Figure 5. -** Visual description of the MS-BD sample. Left panels: brown dwarf masses (top) and eccentricities (bottom) as a function of orbital period for our entire sample (pink, Sect. \ref{sec:Sample}), with confirmed brown dwarfs and false positives found in our follow-up and previous surveys highlighted (Sect. \ref{sec:Contamination}). Black contours correspond to detection probabilities that are marginalised over distance and primary mass as computed in Sect. \ref{app:desert}, and it can be seen that the sample roughly matches the expected distribution from selection effects. Top right panel: MS-BD sample overlayed on the $\gaia$ colour-magnitude diagram, with the density of all sources within 300 pc with XP spectra shown in greyscale. Bottom right panel: stacked distribution of the apparent $\gaia$ G-band magnitude for the sample, showing each step in the selection similar to Fig. \ref{fig:massfunction}. (*fig:SampleDiagnostics*)

</div>
<div id="div_fig3">

<img src="tmp_2611.21800/./figures/metallicity_distribution.png" alt="Fig3" width="100%"/>

**Figure 3. -** Distribution of metallicities [M/H] of our MS-BD systems compared to stellar and planetary companions. Bottom panel: distribution of metallicities in the MS-BD sample (pink) compared to that of all large planets ${>}5$\mjup$$ at astrometrically detectable periods $P\in(50,2000)$ days from the NASA exoplanet archive \citep[gold,][]{Christiansen2025}. This demonstrates the consistency of MS-BD binaries with stellar binaries, and inconsistency with planetary systems. Caution should be taken at the extreme metallicities, as these measurements are more uncertain and the errorbars represent lower bounds. Top panel: rate of MS-BD binaries in bins of metallicity by comparing the MS-BD sample to the representative parent sample (App. \ref{app:parentsample}) shown in pink, compared to the rate of binarity for stellar binaries from [Moe, Kratter and Badenes (2019)](https://ui.adsabs.harvard.edu/abs/2019ApJ...875...61M), where the y-axis is meaningless. (*fig:metallicities*)

</div><div id="qrcode"><img src=https://api.qrserver.com/v1/create-qr-code/?size=100x100&data="https://arxiv.org/abs/2611.21800"></div>

<div class="macros" style="visibility:hidden;">
$\newcommand{\ensuremath}{}$
$\newcommand{\xspace}{}$
$\newcommand{\object}[1]{\texttt{#1}}$
$\newcommand{\farcs}{{.}''}$
$\newcommand{\farcm}{{.}'}$
$\newcommand{\arcsec}{''}$
$\newcommand{\arcmin}{'}$
$\newcommand{\ion}[2]{#1#2}$
$\newcommand{\textsc}[1]{\textrm{#1}}$
$\newcommand{\hl}[1]{\textrm{#1}}$
$\newcommand{\footnote}[1]{}$
$\newcommand{\K}{K}$
$\newcommand{\hw}{\Gamma}$
$\newcommand{\aap}{Astron. Astrophys.}$
$\newcommand{\aaps}{Astron. Astrophys. Suppl. Ser.}$
$\newcommand{\aj}{Astron. J.}$
$\newcommand{\apj}{Astrophys. J.}$
$\newcommand{\jqsrt}{J. Quant. Spectrosc. Radiat. Transf.}$
$\newcommand{\mnras}{Mon. Not. R. Astron. Soc.}$
$\newcommand{\zap}{Z. Astrophys.}$</div>



<div id="title">

# A branchless single-precision Voigt approximation for spectral synthesis$\tnoteref{am}$

</div>
<div id="comments">

[![arXiv](https://img.shields.io/badge/arXiv-2611.22473-b31b1b.svg)](https://arxiv.org/abs/2611.22473)<mark>Appeared on: 2026-11-18</mark> -  _Accepted for publication in J. Quant. Spectrosc. Radiat. Transf. 10 pages, 2 figures, 5 tables. Fortran module implementing the QLG members included as ancillary file_

</div>
<div id="authors">

<mark>R. Hoppe</mark>

</div>
<div id="abstract">

**Abstract:** The Voigt function is evaluated repeatedly in spectral synthesis, and several widely used implementations carry an absolute error floor. We construct QLG, a family of branchless, single-precision approximations, separable in the damping parameter $a$ . QLG is named for its three ingredients: Lorentzian and Gaussian generators $L$ and $G$ , and an envelope $q$ that multiplies every fitted correction and vanishes at both endpoints. The leading Lorentzian wing amplitude is fixed analytically. Three members reach observed maximum full-profile relative errors of $6.5\times10^{-3}$ , $9.2\times10^{-4}$ , and $7\times10^{-5}$ over $a\in[10^{-9},10^{9}]$ and $|v|\le10^{5}$ . Extended-domain and randomized checks support these error levels. For the tested siu , moog , and Humlí $\v{c}$ ek \texttt{w4} implementations, the relative error instead approaches $100\%$ towards small values of the damping parameter. We compare synthesis-weighted costs on an Apple M1 and an AMD EPYC 9554 using \texttt{gfortran 15} , and propagate errors through solar-dwarf, red-supergiant, and M-dwarf syntheses. The least expensive member costs $0.98$ -- $1.05$ times as much as the tabulated Synspec kernel on the M1 and $1.11$ -- $1.20$ times as much on the EPYC, with a maximum flux error smaller by a factor of $3.6$ . The intermediate member costs less than every tested arithmetic kernel and holds the flux error below $10^{-3}$ . The most accurate member costs $1.6$ -- $3.0$ times less than \texttt{w4} and gives $3.4$ -- $5.0$ times smaller maximum flux errors in the tested syntheses.

</div>

<div id="div_fig1">

<img src="tmp_2611.22473/figs/fig5_pseudo.png" alt="Fig1" width="100%"/>

**Figure 1. -** The residual $[F-P]/q$ of Eq. (\ref{eq:qlg_general}) that the fitted correction must represent, over the $(r,L)$ domain, normalized to the profile peak. The colour map is centred on zero, so its orange half covers the small negative range. A single slice at fixed $r$, marked here at $r=0.2$, approximates the residual at every $r$ to within the accuracy required of the lowest tier. That slice is the basis function $Z_{0.2}$ of Eq. (\ref{eq:zshape}), which QLG-bot reads from a table. (*fig:pseudo*)

</div>
<div id="div_fig2">

<img src="tmp_2611.22473/figs/cahk.png" alt="Fig2" width="100%"/>

**Figure 2. -** Solar-dwarf normalized flux around Ca ii H\&K, calculated with HAW in \texttt{fp32}, the \texttt{fp64} reference kernel, and QLG-bot, whose maximum full-profile relative error over the validation domain is $6.5\times10^{-3}$(Table \ref{tab:deployed}). Negative HAW profile values were clamped to zero before their opacities entered the synthesis. QLG-bot, the least accurate of the three members, is visually indistinguishable from the reference here, whereas HAW suppresses the broad damping wings. (*fig:cahk*)

</div><div id="qrcode"><img src=https://api.qrserver.com/v1/create-qr-code/?size=100x100&data="https://arxiv.org/abs/2611.22473"></div>

<div class="macros" style="visibility:hidden;">
$\newcommand{\ensuremath}{}$
$\newcommand{\xspace}{}$
$\newcommand{\object}[1]{\texttt{#1}}$
$\newcommand{\farcs}{{.}''}$
$\newcommand{\farcm}{{.}'}$
$\newcommand{\arcsec}{''}$
$\newcommand{\arcmin}{'}$
$\newcommand{\ion}[2]{#1#2}$
$\newcommand{\textsc}[1]{\textrm{#1}}$
$\newcommand{\hl}[1]{\textrm{#1}}$
$\newcommand{\footnote}[1]{}$
$\newcommand\omc{\omega Cen}$
$\newcommand{\todo}[1]{{\textcolor{red}{TODO: #1}}}$</div>



<div id="title">

# oMEGACat-JWST I.: Project description and photometric data release

</div>
<div id="comments">

[![arXiv](https://img.shields.io/badge/arXiv-2611.23150-b31b1b.svg)](https://arxiv.org/abs/2611.23150)<mark>Appeared on: 2026-11-18</mark> -  _16 pages, 11 Figures; Submitted to Astronomy and Astrophysics_

</div>
<div id="authors">

M. Häberle, et al. -- incl., <mark>C. Clontz</mark>, <mark>S. Souza</mark>, <mark>N. Neumayer</mark>, <mark>P. J. Smith</mark>, <mark>J. E. Davies</mark>, <mark>A. d. Graaff</mark>

</div>
<div id="abstract">

**Abstract:** Omega Centauri (NGC 5139, $\omc$ ) is the most massive globular cluster in the Milky Way and the likely stripped nucleus of an accreted dwarf galaxy. Extensive Hubble Space Telescope (HST) observations have uncovered many of its peculiar features, including complex subpopulations and kinematic evidence for an intermediate-mass black hole via central fast-moving stars. High-precision astrometric follow-up is essential to constrain the orbits of these stars. We present oMEGACat-JWST, a deep photometric and astrometric catalog of $\omc$ 's core and surroundings that extends our previous oMEGACat efforts to near-infrared wavelengths. The superior sensitivity and angular resolution of the James Webb Space Telescope (JWST) provide significant astrometric and photometric improvements, particularly for faint stars. Our catalog is based on a custom reduction of NIRCam and NIRISS imaging across three JWST programs (GO-4343, GO-6154, and GO-8322) extending out to two half-light radii ( $10\arcmin$ ). In total, our catalog contains approximately $677 000$ sources. In the region overlapping with oMEGACat, we achieve high relative completeness ( $>90\%$ of HST sources recovered) while detecting over $86 000$ new sources below HST's detection limit. Astrometric tests demonstrate a single-image precision as good as 0.5 \% of a NIRCam pixel ( $\sim150 \mathrm{\mu as}$ ) at m $_{\rm F115W}\sim18$ ; at  m $_{\rm F115W}\sim23$ the precision is still $\sim$ 1 mas.Color-magnitude diagrams based on our data show the well-known split between the primordial Helium and the Helium enriched populations and are particularly sensitive to light-element (CNO) variations of faint stars. oMEGACat-JWST provides the most complete photometric catalog of $\omc$ 's core to date. Combined with HST data, it enables multi-wavelength subpopulation analyses. Together with future monitoring observations, its high astrometric precision will improve our understanding of the kinematics of the central fast-moving stars in $\omc$ .

</div>

<div id="div_fig1">

<img src="tmp_2611.23150/./jwst_finding_chart_jwst.png" alt="Fig4" width="100%"/>

**Figure 4. -** Footprints of the JWST observations. Two NIRCam programs (GO-4343, GO-8322) cover the central region of the cluster, while parallel imaging with NIRISS and an additional program (GO-6154) cover larger radii. The dataset not presented in this article are marked with "*". The background image is based on ground-based wide-field VST data. We also show an HST-based color image that covers the oMEGACat footprint out to about one half-light radius.\\_Credit VST: ESO/INAF-VST/OmegaCAM. Acknowledgment: A. Grado, L. Limatola/INAF-Capodimonte Observatory_.\\_Credit HST : ESA/Hubble, NASA, Maximilian Häberle (MPIA)_. (*fig:footprints*)

</div>
<div id="div_fig2">

<img src="tmp_2611.23150/./three_panels_two_isochrones.png" alt="Fig8" width="100%"/>

**Figure 8. -** CMD comparison for all 3 fields. Here we show three comparable CMDs based on the different datasets analyzed in this work. Well-measured sources are shown in black, while partially saturated sources are shown in red. The overall CMD morphology and photometric calibration show good agreement, even though photometric errors and saturation thresholds differ. All CMDs show a clear split of the main-sequence and a broadening of the color distribution below the main-sequence knee. For comparison, we also plot a MIST isochrone representing the primordial He (Y=0.25) main population of \omc and another custom isochrone with an enriched He fraction (Y=0.4). These models provide a good fit to the upper part of the main sequence, but do not yet reproduce the the low mass / temperature region. (*fig:3panel_cmd*)

</div>
<div id="div_fig3">

<img src="tmp_2611.23150/./internal_astrometric_consistency.png" alt="Fig2.1" width="50%"/><img src="tmp_2611.23150/./internal_astrometric_differences.png" alt="Fig2.2" width="50%"/>

**Figure 2. -** Astrometric consistency tests. In the upper two panels, we compare how the astrometric scatter (measured as the standard deviation of stars with at least three measurements) depends on magnitude and filter. The three lines indicate the median scatter for the three different SWC configurations in different magnitude bins. Both filters reach an RMS of $\sim0.5$\% of a NIRCam pixel for bright unsaturated stars. \     In the lower plot, we compare how well the positions measured with the two different SWC filters _(left)_ or measured at the two different JWST epochs _(right)_ agree. The agreement between both filters is as good as 0.008 NIRCam pixel (0.15 mas). The larger deviations between the two JWST epochs are caused by proper-motion displacements of the stars and are consistent with the velocity dispersion in \omc. (*fig:astrometric_consistency_jwst*)

</div><div id="qrcode"><img src=https://api.qrserver.com/v1/create-qr-code/?size=100x100&data="https://arxiv.org/abs/2611.23150"></div>

# Create HTML index

In [10]:
from datetime import datetime, timedelta, timezone
from glob import glob
import os

files = glob('_build/html/*.md')
days = 7
now = datetime.today()
res = []
for fk in files:
    stat_result = os.stat(fk).st_ctime
    modified = datetime.fromtimestamp(stat_result, tz=timezone.utc).replace(tzinfo=None)
    delta = now.today() - modified
    if delta <= timedelta(days=days):
        res.append((delta.seconds, fk))
res = [k[1] for k in reversed(sorted(res, key=lambda x:x[1]))]
npub = len(res)
print(len(res), f" publications files modified in the last {days:d} days.")
# [ print('\t', k) for k in res ];

157  publications files modified in the last 7 days.


In [11]:
import datetime
from glob import glob

def get_last_n_days(lst, days=1):
    """ Get the documents from the last n days """
    sorted_lst = sorted(lst, key=lambda x: x[1], reverse=True)
    for fname, date in sorted_lst:
        if date >= str(datetime.date.today() - datetime.timedelta(days=days)):
            yield fname

def extract_appearance_dates(lst_file):
    dates = []

    def get_date(line):
        return line\
            .split('Appeared on:')[-1]\
            .split('</mark>')[0].strip()

    for fname in lst:
        with open(fname, 'r') as f:
            found_date = False
            for line in f:
                if not found_date:
                    if "Appeared on" in line:
                        found_date = True
                        dates.append((fname, get_date(line)))
                else:
                    break
    return dates

from glob import glob
lst = glob('_build/html/*md')
days = 7
dates = extract_appearance_dates(lst)
res = list(get_last_n_days(dates, days))
npub = len(res)
print(len(res), f" publications in the last {days:d} days.")

20  publications in the last 7 days.


In [12]:
def create_carousel(npub=4):
    """ Generate the HTML code for a carousel with `npub` slides """
    carousel = ["""  <div class="carousel" """,
                """       data-flickity='{ "autoPlay": 10000, "adaptiveHeight": true, "resize": true, "wrapAround": true, "pauseAutoPlayOnHover": true, "groupCells": 1 }' id="asyncTypeset">"""
                ]
    
    item_str = """    <div class="carousel-cell"> <div id="slide{k}" class="md_view">Content {k}</div> </div>"""
    for k in range(1, npub + 1):
        carousel.append(item_str.format(k=k))
    carousel.append("  </div>")
    return '\n'.join(carousel)

def create_grid(npub=4):
    """ Generate the HTML code for a flat grid with `npub` slides """
    grid = ["""  <div class="grid"> """,
                ]
    
    item_str = """    <div class="grid-item"> <div id="slide{k}" class="md_view">Content {k}</div> </div>"""
    for k in range(1, npub + 1):
        grid.append(item_str.format(k=k))
    grid.append("  </div>")
    return '\n'.join(grid)

In [13]:
carousel = create_carousel(npub)
docs = ', '.join(['"{0:s}"'.format(k.split('/')[-1]) for k in res])
slides = ', '.join([f'"slide{k}"' for k in range(1, npub + 1)])

with open("daily_template.html", "r") as tpl:
    page = tpl.read()
    page = page.replace("{%-- carousel:s --%}", carousel)\
               .replace("{%-- suptitle:s --%}",  "7-day archives" )\
               .replace("{%-- docs:s --%}", docs)\
               .replace("{%-- slides:s --%}", slides)
    
with open("_build/html/index_7days.html", 'w') as fout:
    fout.write(page)

In [14]:
# redo for today
days = 1
res = list(get_last_n_days(dates, days))
npub = len(res)
print(len(res), f" publications in the last day.")

carousel = create_carousel(npub)
docs = ', '.join(['"{0:s}"'.format(k.split('/')[-1]) for k in res])
slides = ', '.join([f'"slide{k}"' for k in range(1, npub + 1)])

with open("daily_template.html", "r") as tpl:
    page = tpl.read()
    page = page.replace("{%-- carousel:s --%}", carousel)\
               .replace("{%-- suptitle:s --%}",  "Daily" )\
               .replace("{%-- docs:s --%}", docs)\
               .replace("{%-- slides:s --%}", slides)
    
# print(carousel, docs, slides)
# print(page)
with open("_build/html/index_daily.html", 'w') as fout:
    fout.write(page)

8  publications in the last day.


In [15]:
# Create the flat grid of the last N papers (fixed number regardless of dates)
from itertools import islice 

npub = 6
res = [k[0] for k in (islice(reversed(sorted(dates, key=lambda x: x[1])), 6))]
print(len(res), f" {npub} publications selected.")

grid = create_grid(npub)
docs = ', '.join(['"{0:s}"'.format(k.split('/')[-1]) for k in res])
slides = ', '.join([f'"slide{k}"' for k in range(1, npub + 1)])

with open("grid_template.html", "r") as tpl:
    page = tpl.read()
    page = page.replace("{%-- grid-content:s --%}", grid)\
               .replace("{%-- suptitle:s --%}",  f"Last {npub:,d} papers" )\
               .replace("{%-- docs:s --%}", docs)\
               .replace("{%-- slides:s --%}", slides)
    
# print(grid, docs, slides)
# print(page)
with open("_build/html/index_npub_grid.html", 'w') as fout:
    fout.write(page)

6  6 publications selected.
